In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "WorkstationOffer"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

offers = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="cpu",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="gpu",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="price_pln",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="ram_gb",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="vram_gb",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="in_stock",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: WorkstationOffer


In [5]:
workstation_data = [
    {
        "name": "AI Starter Workstation",
        "description": "Entry-level workstation for Python development, small machine learning experiments, Jupyter notebooks and Docker containers.",
        "category": "AI Workstation",
        "cpu": "AMD Ryzen 7",
        "gpu": "NVIDIA RTX 3060",
        "price_pln": 5200.0,
        "ram_gb": 32,
        "vram_gb": 12,
        "in_stock": True,
    },
    {
        "name": "AI Pro Workstation",
        "description": "Powerful workstation for local LLM experiments, CUDA workloads, computer vision, vector databases and data science projects.",
        "category": "AI Workstation",
        "cpu": "Intel Xeon",
        "gpu": "NVIDIA RTX 4000 Ada",
        "price_pln": 7900.0,
        "ram_gb": 64,
        "vram_gb": 20,
        "in_stock": True,
    },
    {
        "name": "Data Science Laptop",
        "description": "Portable machine for Python notebooks, pandas, scikit-learn, lightweight model training and remote development.",
        "category": "Laptop",
        "cpu": "Intel Core Ultra 7",
        "gpu": "NVIDIA RTX 4060 Laptop",
        "price_pln": 6900.0,
        "ram_gb": 32,
        "vram_gb": 8,
        "in_stock": True,
    },
    {
        "name": "Office Mini PC",
        "description": "Compact office computer for email, documents, web applications, video meetings and light scripting tasks.",
        "category": "Office PC",
        "cpu": "Intel Core i5",
        "gpu": "Integrated graphics",
        "price_pln": 2600.0,
        "ram_gb": 16,
        "vram_gb": 0,
        "in_stock": True,
    },
    {
        "name": "Deep Learning Tower",
        "description": "High-end tower for deep learning, large models, GPU acceleration, CUDA development and heavy AI workloads.",
        "category": "AI Workstation",
        "cpu": "AMD Threadripper",
        "gpu": "NVIDIA RTX 4090",
        "price_pln": 15500.0,
        "ram_gb": 128,
        "vram_gb": 24,
        "in_stock": False,
    },
    {
        "name": "Backend Developer PC",
        "description": "Reliable development machine for FastAPI, Docker, PostgreSQL, testing, CI simulation and daily backend engineering.",
        "category": "Developer Workstation",
        "cpu": "AMD Ryzen 9",
        "gpu": "Integrated graphics",
        "price_pln": 6100.0,
        "ram_gb": 64,
        "vram_gb": 0,
        "in_stock": True,
    },
]

In [6]:
result = offers.data.insert_many(workstation_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = offers.query.fetch_objects(
    limit=10,
    return_properties=[
        "name",
        "category",
        "cpu",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "in_stock",
    ],
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 80)

832bc603-8764-4c08-af6e-f933d25f163d
{'category': 'Office PC', 'name': 'Office Mini PC', 'gpu': 'Integrated graphics', 'ram_gb': 16, 'price_pln': 2600.0, 'cpu': 'Intel Core i5', 'vram_gb': 0, 'in_stock': True}
--------------------------------------------------------------------------------
8cbfa559-d730-402f-816c-8d37638254d1
{'category': 'AI Workstation', 'name': 'AI Pro Workstation', 'in_stock': True, 'ram_gb': 64, 'price_pln': 7900.0, 'cpu': 'Intel Xeon', 'vram_gb': 20, 'gpu': 'NVIDIA RTX 4000 Ada'}
--------------------------------------------------------------------------------
9685c2de-09af-4b81-86e3-44cd3cff6ed4
{'name': 'Deep Learning Tower', 'category': 'AI Workstation', 'gpu': 'NVIDIA RTX 4090', 'ram_gb': 128, 'in_stock': False, 'cpu': 'AMD Threadripper', 'vram_gb': 24, 'price_pln': 15500.0}
--------------------------------------------------------------------------------
ae9a3c6f-cd91-4a6d-9aa0-d6c1e1ea2d1c
{'category': 'AI Workstation', 'name': 'AI Starter Workstation', 'in_s

In [8]:
response = offers.query.near_text(
    query="computer for local AI experiments and machine learning",
    limit=5,
    return_properties=[
        "name",
        "description",
        "category",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "in_stock",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Name:", obj.properties["name"])
    print("GPU:", obj.properties["gpu"])
    print("Price:", obj.properties["price_pln"])
    print("RAM:", obj.properties["ram_gb"])
    print("VRAM:", obj.properties["vram_gb"])
    print("In stock:", obj.properties["in_stock"])
    print("-" * 80)

Distance: 0.38649410009384155
Name: AI Pro Workstation
GPU: NVIDIA RTX 4000 Ada
Price: 7900.0
RAM: 64
VRAM: 20
In stock: True
--------------------------------------------------------------------------------
Distance: 0.432170033454895
Name: AI Starter Workstation
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
In stock: True
--------------------------------------------------------------------------------
Distance: 0.45648717880249023
Name: Data Science Laptop
GPU: NVIDIA RTX 4060 Laptop
Price: 6900.0
RAM: 32
VRAM: 8
In stock: True
--------------------------------------------------------------------------------
Distance: 0.48134732246398926
Name: Deep Learning Tower
GPU: NVIDIA RTX 4090
Price: 15500.0
RAM: 128
VRAM: 24
In stock: False
--------------------------------------------------------------------------------
Distance: 0.5332958698272705
Name: Backend Developer PC
GPU: Integrated graphics
Price: 6100.0
RAM: 64
VRAM: 0
In stock: True
---------------------------------------------

In [9]:
response = offers.query.near_text(
    query="workstation for local AI and CUDA workloads",
    filters=Filter.by_property("price_pln").less_or_equal(8000),
    limit=5,
    return_properties=[
        "name",
        "description",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "in_stock",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Name:", obj.properties["name"])
    print("GPU:", obj.properties["gpu"])
    print("Price:", obj.properties["price_pln"])
    print("RAM:", obj.properties["ram_gb"])
    print("VRAM:", obj.properties["vram_gb"])
    print("In stock:", obj.properties["in_stock"])
    print("-" * 80)

Distance: 0.24396055936813354
Name: AI Pro Workstation
GPU: NVIDIA RTX 4000 Ada
Price: 7900.0
RAM: 64
VRAM: 20
In stock: True
--------------------------------------------------------------------------------
Distance: 0.3141753077507019
Name: AI Starter Workstation
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
In stock: True
--------------------------------------------------------------------------------
Distance: 0.4036921262741089
Name: Data Science Laptop
GPU: NVIDIA RTX 4060 Laptop
Price: 6900.0
RAM: 32
VRAM: 8
In stock: True
--------------------------------------------------------------------------------
Distance: 0.4236786365509033
Name: Backend Developer PC
GPU: Integrated graphics
Price: 6100.0
RAM: 64
VRAM: 0
In stock: True
--------------------------------------------------------------------------------
Distance: 0.5684778690338135
Name: Office Mini PC
GPU: Integrated graphics
Price: 2600.0
RAM: 16
VRAM: 0
In stock: True
---------------------------------------------------

In [10]:
filters = (
    Filter.by_property("price_pln").less_or_equal(8000)
    & Filter.by_property("ram_gb").greater_or_equal(32)
    & Filter.by_property("vram_gb").greater_or_equal(8)
    & Filter.by_property("in_stock").equal(True)
)

response = offers.query.near_text(
    query="best machine for local LLMs and machine learning experiments",
    filters=filters,
    limit=5,
    return_properties=[
        "name",
        "description",
        "category",
        "cpu",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "in_stock",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Name:", obj.properties["name"])
    print("Category:", obj.properties["category"])
    print("CPU:", obj.properties["cpu"])
    print("GPU:", obj.properties["gpu"])
    print("Price:", obj.properties["price_pln"])
    print("RAM:", obj.properties["ram_gb"])
    print("VRAM:", obj.properties["vram_gb"])
    print("In stock:", obj.properties["in_stock"])
    print("-" * 80)

Distance: 0.4793059825897217
Name: AI Pro Workstation
Category: AI Workstation
CPU: Intel Xeon
GPU: NVIDIA RTX 4000 Ada
Price: 7900.0
RAM: 64
VRAM: 20
In stock: True
--------------------------------------------------------------------------------
Distance: 0.5270360112190247
Name: Data Science Laptop
Category: Laptop
CPU: Intel Core Ultra 7
GPU: NVIDIA RTX 4060 Laptop
Price: 6900.0
RAM: 32
VRAM: 8
In stock: True
--------------------------------------------------------------------------------
Distance: 0.5388427972793579
Name: AI Starter Workstation
Category: AI Workstation
CPU: AMD Ryzen 7
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
In stock: True
--------------------------------------------------------------------------------


In [11]:
filters = (
    Filter.by_property("price_pln").less_or_equal(8000)
    & Filter.by_property("in_stock").equal(True)
)

response = offers.query.hybrid(
    query="RTX workstation for AI",
    alpha=0.5,
    filters=filters,
    limit=5,
    return_properties=[
        "name",
        "description",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Name:", obj.properties["name"])
    print("GPU:", obj.properties["gpu"])
    print("Price:", obj.properties["price_pln"])
    print("RAM:", obj.properties["ram_gb"])
    print("VRAM:", obj.properties["vram_gb"])
    print("-" * 80)

Score: 0.9921518564224243
Name: AI Pro Workstation
GPU: NVIDIA RTX 4000 Ada
Price: 7900.0
RAM: 64
VRAM: 20
--------------------------------------------------------------------------------
Score: 0.9496015310287476
Name: AI Starter Workstation
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
--------------------------------------------------------------------------------
Score: 0.32408565282821655
Name: Backend Developer PC
GPU: Integrated graphics
Price: 6100.0
RAM: 64
VRAM: 0
--------------------------------------------------------------------------------
Score: 0.27017369866371155
Name: Data Science Laptop
GPU: NVIDIA RTX 4060 Laptop
Price: 6900.0
RAM: 32
VRAM: 8
--------------------------------------------------------------------------------
Score: 0.0
Name: Office Mini PC
GPU: Integrated graphics
Price: 2600.0
RAM: 16
VRAM: 0
--------------------------------------------------------------------------------


In [12]:
def recommend_workstation(
    query: str,
    *,
    max_price: float,
    min_ram_gb: int = 16,
    min_vram_gb: int = 0,
    only_in_stock: bool = True,
    limit: int = 5,
) -> None:
    filters = (
        Filter.by_property("price_pln").less_or_equal(max_price)
        & Filter.by_property("ram_gb").greater_or_equal(min_ram_gb)
        & Filter.by_property("vram_gb").greater_or_equal(min_vram_gb)
    )

    if only_in_stock:
        filters = filters & Filter.by_property("in_stock").equal(True)

    response = offers.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "name",
            "description",
            "category",
            "cpu",
            "gpu",
            "price_pln",
            "ram_gb",
            "vram_gb",
            "in_stock",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("MAX PRICE:", max_price)
    print("MIN RAM:", min_ram_gb)
    print("MIN VRAM:", min_vram_gb)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Name:", obj.properties["name"])
        print("Category:", obj.properties["category"])
        print("CPU:", obj.properties["cpu"])
        print("GPU:", obj.properties["gpu"])
        print("Price:", obj.properties["price_pln"])
        print("RAM:", obj.properties["ram_gb"])
        print("VRAM:", obj.properties["vram_gb"])
        print("In stock:", obj.properties["in_stock"])
        print("Description:", obj.properties["description"])
        print("-" * 80)

In [13]:
recommend_workstation(
    "I need a workstation for local AI experiments and LLM inference",
    max_price=8000,
    min_ram_gb=32,
    min_vram_gb=8,
)

QUERY: I need a workstation for local AI experiments and LLM inference
MAX PRICE: 8000
MIN RAM: 32
MIN VRAM: 8
--------------------------------------------------------------------------------
Distance: 0.28514885902404785
Name: AI Pro Workstation
Category: AI Workstation
CPU: Intel Xeon
GPU: NVIDIA RTX 4000 Ada
Price: 7900.0
RAM: 64
VRAM: 20
In stock: True
Description: Powerful workstation for local LLM experiments, CUDA workloads, computer vision, vector databases and data science projects.
--------------------------------------------------------------------------------
Distance: 0.3555530905723572
Name: AI Starter Workstation
Category: AI Workstation
CPU: AMD Ryzen 7
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
In stock: True
Description: Entry-level workstation for Python development, small machine learning experiments, Jupyter notebooks and Docker containers.
--------------------------------------------------------------------------------
Distance: 0.4306749701499939
Name: D

In [14]:
recommend_workstation(
    "I need a computer for FastAPI Docker and backend development",
    max_price=7000,
    min_ram_gb=32,
    min_vram_gb=0,
)

QUERY: I need a computer for FastAPI Docker and backend development
MAX PRICE: 7000
MIN RAM: 32
MIN VRAM: 0
--------------------------------------------------------------------------------
Distance: 0.3298248052597046
Name: Backend Developer PC
Category: Developer Workstation
CPU: AMD Ryzen 9
GPU: Integrated graphics
Price: 6100.0
RAM: 64
VRAM: 0
In stock: True
Description: Reliable development machine for FastAPI, Docker, PostgreSQL, testing, CI simulation and daily backend engineering.
--------------------------------------------------------------------------------
Distance: 0.47845637798309326
Name: AI Starter Workstation
Category: AI Workstation
CPU: AMD Ryzen 7
GPU: NVIDIA RTX 3060
Price: 5200.0
RAM: 32
VRAM: 12
In stock: True
Description: Entry-level workstation for Python development, small machine learning experiments, Jupyter notebooks and Docker containers.
--------------------------------------------------------------------------------
Distance: 0.5759339332580566
Name: Data

In [15]:
recommend_workstation(
    "I need a cheap office machine for email and documents",
    max_price=3500,
    min_ram_gb=8,
    min_vram_gb=0,
)

QUERY: I need a cheap office machine for email and documents
MAX PRICE: 3500
MIN RAM: 8
MIN VRAM: 0
--------------------------------------------------------------------------------
Distance: 0.4549295902252197
Name: Office Mini PC
Category: Office PC
CPU: Intel Core i5
GPU: Integrated graphics
Price: 2600.0
RAM: 16
VRAM: 0
In stock: True
Description: Compact office computer for email, documents, web applications, video meetings and light scripting tasks.
--------------------------------------------------------------------------------


In [16]:
filters = (
    Filter.by_property("price_pln").less_or_equal(8000)
    & Filter.by_property("ram_gb").greater_or_equal(32)
    & Filter.by_property("vram_gb").greater_or_equal(8)
    & Filter.by_property("in_stock").equal(True)
)

response = offers.generate.near_text(
    query="best workstation for local AI, Python notebooks, vector databases and small LLM experiments",
    filters=filters,
    grouped_task=(
        "Recommend the best option from the retrieved workstation offers. "
        "Explain briefly why it is a good fit. "
        "Mention price, RAM, GPU and VRAM. "
        "Use only the retrieved offers."
    ),
    limit=5,
    return_properties=[
        "name",
        "description",
        "category",
        "cpu",
        "gpu",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "in_stock",
    ],
)

print(response.generated)

print("\nRetrieved offers:")
for obj in response.objects:
    print("-", obj.properties["name"], "|", obj.properties["price_pln"], "PLN")

Based on the retrieved offers, the **AI Pro Workstation** is the best option. 

**Specifications:**
- **Price:** 7900 PLN
- **RAM:** 64 GB
- **GPU:** NVIDIA RTX 4000 Ada
- **VRAM:** 20 GB

### Reasons for Recommendation:
1. **High Performance:** The Intel Xeon CPU combined with the NVIDIA RTX 4000 Ada GPU makes this workstation highly suitable for demanding tasks like local LLM experiments, CUDA workloads, and computer vision projects. This is especially important for AI and data science applications where processing power is critical.

2. **Ample RAM:** With 64 GB of RAM, this workstation can handle larger datasets and multi-threaded tasks efficiently, making it ideal for intensive data processing and machine learning experiments.

3. **Significant VRAM:** The 20 GB VRAM on the GPU provides a larger memory pool for graphics and computation tasks, allowing for more complex models to be run without running into memory constraints.

4. **Future-Proofing:** As AI and machine learning work

In [17]:
def ask_recommender(
    question: str,
    *,
    max_price: float,
    min_ram_gb: int = 16,
    min_vram_gb: int = 0,
    limit: int = 5,
) -> None:
    filters = (
        Filter.by_property("price_pln").less_or_equal(max_price)
        & Filter.by_property("ram_gb").greater_or_equal(min_ram_gb)
        & Filter.by_property("vram_gb").greater_or_equal(min_vram_gb)
        & Filter.by_property("in_stock").equal(True)
    )

    response = offers.generate.near_text(
        query=question,
        filters=filters,
        grouped_task=(
            "Answer as a hardware recommendation assistant. "
            "Use only the retrieved offers. "
            "Recommend the most suitable option and explain briefly why. "
            "Mention important specs such as CPU, GPU, RAM, VRAM and price. "
            "If the retrieved offers are not suitable, say so clearly."
        ),
        limit=limit,
        return_properties=[
            "name",
            "description",
            "category",
            "cpu",
            "gpu",
            "price_pln",
            "ram_gb",
            "vram_gb",
            "in_stock",
        ],
    )

    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.generated)

    print("\nOFFERS USED:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["name"],
            "|",
            obj.properties["gpu"],
            "|",
            obj.properties["ram_gb"],
            "GB RAM |",
            obj.properties["vram_gb"],
            "GB VRAM |",
            obj.properties["price_pln"],
            "PLN",
        )

In [18]:
ask_recommender(
    "What should I buy for local AI experiments and vector database work?",
    max_price=8000,
    min_ram_gb=32,
    min_vram_gb=8,
)

QUESTION:
What should I buy for local AI experiments and vector database work?

ANSWER:
Based on the retrieved offers, the **AI Pro Workstation** is the most suitable option for your needs. Here’s why:

- **CPU:** Intel Xeon - This is a powerful processor designed for demanding tasks, making it ideal for local LLM experiments and CUDA workloads.
- **GPU:** NVIDIA RTX 4000 Ada - This high-performance GPU provides excellent support for AI workloads and data visualization tasks.
- **RAM:** 64 GB - Abundant memory that allows for handling large datasets and training complex models efficiently.
- **VRAM:** 20 GB - High VRAM capacity which is essential for intensive graphical and AI computations.
- **Price:** 7900 PLN - While it is the most expensive option, its specs justify the investment for serious machine learning and data science projects.

Overall, the AI Pro Workstation is tailored for extensive machine learning applications and heavy computational tasks, making it the best choice fo

In [19]:
ask_recommender(
    "What should I buy for backend Python development with Docker and tests?",
    max_price=7000,
    min_ram_gb=32,
    min_vram_gb=0,
)

QUESTION:
What should I buy for backend Python development with Docker and tests?

ANSWER:
Based on the retrieved offers, I recommend the **AI Starter Workstation** for your needs.

### Specifications:
- **CPU**: AMD Ryzen 7
- **GPU**: NVIDIA RTX 3060
- **RAM**: 32 GB
- **VRAM**: 12 GB
- **Price**: 5200 PLN

### Explanation:
The AI Starter Workstation is an excellent choice for anyone involved in Python development, small machine learning experiments, and working with Docker containers. The AMD Ryzen 7 CPU provides solid performance for development tasks, while the NVIDIA RTX 3060 GPU allows for greater computational capabilities, particularly for tasks involving machine learning and data processing.

With 32 GB of RAM, this workstation can handle multitasking and memory-intensive applications effectively. Additionally, the 12 GB of VRAM from the GPU is advantageous for running more complex models and larger datasets.

At a price of 5200 PLN, this workstation offers a good balance of s

In [20]:
ask_recommender(
    "What should I buy for office work and video meetings?",
    max_price=3500,
    min_ram_gb=8,
    min_vram_gb=0,
)

QUESTION:
What should I buy for office work and video meetings?

ANSWER:
The best option retrieved is the **Office Mini PC**. Here are the important specifications and details:

- **CPU**: Intel Core i5
- **GPU**: Integrated graphics
- **RAM**: 16 GB
- **VRAM**: 0 GB (shared with system RAM)
- **Price**: 2600 PLN

**Recommendation Reason**: This compact office computer is suitable for tasks such as email, document work, web applications, video meetings, and light scripting. The 16 GB of RAM ensures smooth multitasking, making it a good fit for typical office needs. However, keep in mind that the integrated graphics may limit performance in demanding applications or gaming. Overall, for light to moderate use, this PC offers a solid balance of performance and price.

OFFERS USED:
- Office Mini PC | Integrated graphics | 16 GB RAM | 0 GB VRAM | 2600.0 PLN


In [21]:
response = offers.query.near_text(
    query="workstation for local AI and LLM experiments",
    filters=Filter.by_property("in_stock").equal(True),
    limit=10,
    return_properties=[
        "name",
        "description",
        "price_pln",
        "ram_gb",
        "vram_gb",
        "gpu",
    ],
    return_metadata=MetadataQuery(distance=True),
)

ranked = []

for obj in response.objects:
    props = obj.properties

    business_score = (
        props["ram_gb"] * 0.1
        + props["vram_gb"] * 1.5
        - props["price_pln"] / 5000
    )

    ranked.append(
        {
            "name": props["name"],
            "gpu": props["gpu"],
            "price_pln": props["price_pln"],
            "ram_gb": props["ram_gb"],
            "vram_gb": props["vram_gb"],
            "distance": obj.metadata.distance,
            "business_score": round(business_score, 3),
        }
    )

ranked = sorted(ranked, key=lambda item: item["business_score"], reverse=True)

for item in ranked:
    print(item)

{'name': 'AI Pro Workstation', 'gpu': 'NVIDIA RTX 4000 Ada', 'price_pln': 7900.0, 'ram_gb': 64, 'vram_gb': 20, 'distance': 0.3235745429992676, 'business_score': 34.82}
{'name': 'AI Starter Workstation', 'gpu': 'NVIDIA RTX 3060', 'price_pln': 5200.0, 'ram_gb': 32, 'vram_gb': 12, 'distance': 0.39515596628189087, 'business_score': 20.16}
{'name': 'Data Science Laptop', 'gpu': 'NVIDIA RTX 4060 Laptop', 'price_pln': 6900.0, 'ram_gb': 32, 'vram_gb': 8, 'distance': 0.48484718799591064, 'business_score': 13.82}
{'name': 'Backend Developer PC', 'gpu': 'Integrated graphics', 'price_pln': 6100.0, 'ram_gb': 64, 'vram_gb': 0, 'distance': 0.5047481060028076, 'business_score': 5.18}
{'name': 'Office Mini PC', 'gpu': 'Integrated graphics', 'price_pln': 2600.0, 'ram_gb': 16, 'vram_gb': 0, 'distance': 0.6012613773345947, 'business_score': 1.08}


In [22]:
client.close()